In [5]:
import matplotlib.pyplot as plt
import pandas as pd
import geopandas as gpd
from shapely.ops import linemerge, split
from shapely.geometry import Point, MultiPoint, LineString, MultiLineString, GeometryCollection

# **Generating the static road structure**

## **Loading road data**
Load **Open Streetmap** Road Data from the shapefile. 

This doesn't change between graphs.

In [6]:
road_labels = ['ST_FULLNAM','SHAPE_Leng','CE_SPEED_L', 'geometry']

roads = gpd.read_file("../OSM_shapefiles/Loudoun_Street_Centerline.shp")    # Load R
roads = roads[roads['CE_SPEED_L'] > 25]
roads = roads[road_labels]

# Networkx

## Find **intersections** and **road segments** 


*Dissolve* roads by street name: Create smaller geometries that represent the street objects

Iterate over each street. Each street can have 
* Multiple, non-continuous lines (MultiLineString) - We merge them
* One singular line (LineString)

### **1. Getting the intersections**
   
Get a bounding box around the geometry, and get which streets are in that box. (This is so we don't search all other streets)

Get intersections by checking if the current street and all other streets match. 

### **2. Getting the road segments**

   You can split a LineString by points with geopandas's *split* function

### **3. Getting endpoints**

   For each street, we need to get endpoints so our graph is full. 
   
Everything is stored in two dictionaries. 

Convert dictionaries into a geopandas dataframe. 

Remove the duplicate intersections by lat/long coordinates aswell. 


In [7]:
# Add intersections and endpoints together. 



roads = roads.dissolve(by="ST_FULLNAM").reset_index()


sindex = roads.sindex #RTree
intersections_dict = {'geometry': []}   # Points with no features, just geometry
segments_list = []                      

# Get interections
for idx, entry in roads.iterrows():
    geom = entry['geometry']
    st_name = entry['ST_FULLNAM']
        
    #If a street is not continuous, merge them into one geometry
    if isinstance(geom, MultiLineString):
        geom = linemerge(geom)

    # Use spatial index to check which streets are in its bounding box        
    matches_idx = list(sindex.intersection(geom.bounds))
    matches = roads.iloc[matches_idx]
    
    # Get all other streets and merge them
    other_roads = matches[matches['ST_FULLNAM'] != st_name].geometry.union_all()
    inters = geom.intersection(other_roads)
    # Returns POINT (1 intersection), MULTIPOINT (2+), GEOMETRYCOLLECTION (EMPTY), LINE (ignore) 
    
    # Store intersections
    if isinstance(inters, Point):
        intersections_dict['geometry'].append(inters)
    elif isinstance(inters, MultiPoint):
        intersections_dict['geometry'].extend(inters.geoms)   # For 1 multipoint of X points, it adds X points. 
    else:
        continue

    # Once we found intersections for this street, split it by the intersections
    # if inters.is_empty: # 0 intersections
    #     line_segments = [geom]
    # else:                
    line_segments = list(split(geom, inters).geoms)

    for segment in line_segments:
        segment_dict = entry.to_dict()  # temporary dict
        segment_dict['geometry'] = segment
        segments_list.append(segment_dict)
    
    # Now add any endpoints the segment may have.
        endpoints = [Point(segment.coords[0]), Point(segment.coords[-1])]
        for endpoint in endpoints:
            intersections_dict['geometry'].append(endpoint)
       
           
intersections = gpd.GeoDataFrame(intersections_dict, geometry='geometry', crs=roads.crs)
road_segments = gpd.GeoDataFrame(segments_list, geometry='geometry', crs=roads.crs)
road_segments["seg_id"] = road_segments.index.astype(str)
intersections["intersection_idx"] = intersections.reset_index().index

print(len(road_segments))
print(len(intersections))


## Removes duplicate nodes from the endpoints/intersection collisions. 
intersections = intersections.drop_duplicates(subset=['geometry'])
intersections = intersections.reset_index(drop=True)
intersections["intersection_idx"] = intersections.index

print(f"Unique intersections: {len(intersections)}")


3384
10033
Unique intersections: 2305


## Finding the roads that overlap each center node. 

Spatial Join 1: Find which roads intersect here. Use a buffer aswell. 

In [8]:
# Spatially join intersections and roads to find which roads intersect.
# buffer

road_segments = road_segments.to_crs(epsg=4326)
intersections = intersections.to_crs(epsg=4326)
# Shows what streets intersect with each point. 
road_groups = gpd.sjoin(
    intersections,  # points
    road_segments,          # lines
    how="left",     
    predicate="intersects"
).drop_duplicates()
road_groups.head(10)
#intersections.head(10)

,geometry,intersection_idx,index_right,ST_FULLNAM,SHAPE_Leng,CE_SPEED_L,seg_id
0,POINT (-77.76858 39.12981),0,1435.0,LEESBURG PIKE,1580.529368,55.0,1435
0,POINT (-77.76858 39.12981),0,0.0,AIRMONT RD,7210.650010,45.0,0
0,POINT (-77.76858 39.12981),0,1436.0,LEESBURG PIKE,1580.529368,55.0,1436
1,POINT (-77.76864 39.12957),1,1464.0,LEESBURG PIKE,1580.529368,55.0,1464
1,POINT (-77.76864 39.12957),1,1.0,AIRMONT RD,7210.650010,45.0,1
1,POINT (-77.76864 39.12957),1,0.0,AIRMONT RD,7210.650010,45.0,0
1,POINT (-77.76864 39.12957),1,1465.0,LEESBURG PIKE,1580.529368,55.0,1465
2,POINT (-77.80953 39.05771),2,6.0,AIRMONT RD,7210.650010,45.0,6
2,POINT (-77.80953 39.05771),2,5.0,AIRMONT RD,7210.650010,45.0,5
2,POINT (-77.80953 39.05771),2,918.0,EBENEZER CHURCH RD,7384.054427,35.0,918


# **Processing the Crashes**

## Weekly Snapshots

Each graph will represent a weekly snapshot of crashes. 

* We get a window of crash data
* J





## Aggregation
Because 


# **Graph Construction**

Use Networkx as our main tool.

We could also store the graphs in a database.

In [ ]:


import networkx as nx
import matplotlib.pyplot as plt
# Relies on current intersection nodes. 
def getGraph(): 
    G = nx.Graph()
    # Add intersection/endpoint nodes to the Graph. 
    # However, this preserves only geometric info, we need to snap crash locations to the nearest intersection. 
    # There will be a step before this. 

    for idx, intersection in intersections.iterrows():
        node_id = intersection['intersection_idx']
        node_attributes = intersection.to_dict()
        G.add_node(node_id, pos=(intersection.geometry.x, intersection.geometry.y), **node_attributes)
    
    # Add Edges to our graph
    # iterate over the road segments

    for seg_id in road_segments['seg_id'].unique():
        # To get the node ids that connect this segment, 
        connecting_intersections = road_groups[road_groups['seg_id'] == seg_id]['intersection_idx'].tolist()
        #print(f"Segment {seg_id}: connected: {connecting_intersections}")
        if(len(connecting_intersections) == 2):
            node1 = connecting_intersections[0]
            node2 = connecting_intersections[1]
            seg_data = road_segments[road_segments['seg_id'] == seg_id].iloc[0]
            G.add_edge(node1, node2, seg_id=seg_id,
                  street_name=seg_data['ST_FULLNAM'],
                  speed_limit=seg_data['CE_SPEED_L'],
                  length=seg_data.geometry.length,
                  geometry=seg_data.geometry)
        
    
    # Get node positions from the graph
    pos = nx.get_node_attributes(G, "pos")

    weekGraph = nx.draw(G, pos=pos, node_color="red", node_size=10, edge_color='gray', width=0.5)
    return weekGraph


In [76]:
import numpy as np
crashes = pd.read_csv('../loudoun_county.csv')
crashes["Crash Date"] = pd.to_datetime(crashes["Crash Date"]) # Parse into dates that pandas can read
crashes['geometry'] = crashes.apply(lambda row:Point(row['x'], row['y']), axis=1)   
crashes = crashes.to_crs(epsg=2284)
intersections = intersections.to_crs(crashes.crs)

numWeeks = 4   # year
# weeklyCounts = []
weekGraphs = []   # Array of graphs.

timestamp = pd.Timestamp("2021-11-18")
for i in range(numWeeks):
    start_week = timestamp
    end_week = start_week + pd.Timedelta(days=7)
    weekCrash = crashes[(crashes['Crash Date'] >= start_week) & (crashes["Crash Date"] < end_week)]
    graph = getNetworxGraph(weekCrash)
    
    timestamp = end_week 

 # Filter down to ONE week
start_week = pd.Timestamp("2024-11-18")
end_week = start_week + pd.Timedelta(days=7)


#
# Input: Pandas crashes from a week
#
def getNetworxGraph(weekCrashes):
    # Always reset.
    intersections['total_crashes'] = 0
    intersections['O_crashes'] = 0
    
    
    # Spatial Join
    crashesToNode = gpd.sjoin_nearest(             # Shows how close crash is to a node (m)
        weekCrashes, # left
        intersections, # right
        how='left',
        distance_col='dist_to_node')
    crashesToNode = crashesToNode.sort_values(by="intersection_idx")
    
#Aggregate sums Crash Severity O
    for i in range(len(intersections)):
        currGroup = crashesToNode[crashesToNode['intersection_idx'] == i]
        if(len(currGroup) != 0):
            total_crashes = len(currGroup)
            O_counts = (currGroup['Crash Severity'] == 'O').sum()
            intersections.iloc[i, intersections.columns.get_loc('total_crashes')] = total_crashes
            intersections.iloc[i, intersections.columns.get_loc('O_crashes')] = O_counts
            # Intersection nodes now have total_crashes and O_crashes.
            return getGraph()
            
            





## Add this data to the nodes/intersections dict.

C:\Users\liamm\AppData\Local\Temp\ipykernel_14228\2791733588.py:2: DtypeWarning: Columns (116,118) have mixed types. Specify dtype option on import or set low_memory=False.
  crashes = pd.read_csv('../loudoun_county.csv')


95


,geometry,intersection_idx,total_crashes,O_crashes
0,POINT (11690495.856 4300025.217),0,0,0
1,POINT (11690478.77 4299936.802),1,0,0
2,POINT (11679066.286 4273668.408),2,0,0
3,POINT (11685480.867 4283588.01),3,0,0
4,POINT (11690411.667 4299612.097),4,0,0
5,POINT (11689936.161 4295824.075),5,0,0
6,POINT (11684069.45 4279740.004),6,0,0
7,POINT (11675739.782 4272087.461),7,0,0
8,POINT (11674858.991 4271633.264),8,0,0
9,POINT (11672096.332 4269238.438),9,0,0


# **Optional Folium Exploration**

In [81]:
import numpy as np
from IPython.display import display

# 1. Base Map with Road Segments
m = road_segments.explore(
    column="seg_id",
    categorical=True,
    tooltip=True,
    legend=False,
    style_kwds={"weight": 2} # Make roads slightly thinner to see nodes better
)

# 2. Intersections as black background circles
intersections.explore(
    m=m,
    marker_type="circle",
    tooltip=["intersection_idx", "total_crashes", "O_crashes"], # Show your new stats!
    marker_kwds={
        "radius": 5, 
        "fill": True, 
        "fillColor": "black", 
        "fillOpacity": 0.7,
        "color": "black"
    }
)

# 3. Crashes Layer
# We can color them based on severity to see the 'O' types clearly
crashes.explore(
    m=m,
    column="Crash Severity", # Colors points based on the severity column
    cmap=['red', 'yellow', 'orange', 'green','blue'], # Custom colors (O will likely be the first)
    marker_type="circle_marker",
    marker_kwds={
        "radius": 4,
        "fill": True,
        "fillOpacity": 0.8
    }
)

m.save("nodes.html")